# Golden Hour — Guardian V2 FINAL Verification

**This is a self-contained verification and release notebook.** It implements the reviewed all-currency Guardian V2 logic, adds an exact-byte read-time SHA-256 snapshot of `account_risk.csv`, runs the full account selection with hidden and shuffled evaluation labels, checks independent union accounting for every baseline, and produces input/output provenance.

**Never overwrites the verified team files.** Each run writes into a new `guardian_v2_final_verification_<UTCtime>_<suffix>/` directory under `/home/ibm-user/shared`. The existing `/shared/freeze_plan.csv`, current V2 candidate file, Ahmed explanations, and all other notebooks stay untouched.

For Alex-independent verification, fill the optional `ALEX_EXPECTED_SHA256` (provided by Alex independently), and/or `ALEX_REFERENCE_PATH` (a separately exported, verified detector scores CSV). Otherwise those checks remain **UNVERIFIED**.

This is an all-currency retrospective **review-only recommendation**. No transfers are blocked, no actual money recovered, and model scores may use full-history features.

**How to run:** Upload this notebook into `/home/ibm-user/shared`, open it, choose Kernel → Restart Kernel and Run All Cells. Inspect the printed summary and open the generated `guardian_v2_verification_report.md`.


In [ ]:
# ===== ORIGINAL V2, CODE CELL 2 (read-time provenance patch only) =====
from __future__ import annotations
import os
for _var in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import hashlib, json, time, io
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix, vstack
try:
    from scipy.optimize import milp, Bounds, LinearConstraint
    HAVE_MILP = True
except ImportError:
    HAVE_MILP = False

# Approximate USD for one original-currency unit, a fixed prototype reference.
# In particular, this Bitcoin conversion is not a live or time-specific rate.
FX_TO_USD = {
    'US Dollar': 1.0, 'Euro': 1.08, 'UK Pound': 1.27, 'Swiss Franc': 1.12,
    'Yen': 0.0067, 'Yuan': 0.14, 'Rupee': 0.012, 'Ruble': 0.011,
    'Canadian Dollar': 0.73, 'Australian Dollar': 0.66, 'Mexican Peso': 0.055,
    'Brazil Real': 0.18, 'Saudi Riyal': 0.27, 'Shekel': 0.27,
    'Bitcoin': 60000.0,
}

@dataclass(frozen=True)
class Config:
    k: int = 10
    min_account_score: float = 0.10
    max_candidates: int = 65
    shortlist_each: int = 14
    candidate_lambdas: tuple = (0., 1., 3., 5., 10., 20., 50.)
    frontier_lambdas: tuple = (0., 2., 5., 10., 20., 50.)
    chosen_lambda: float = 20.0  # tested as a sensitivity setting, not default selection
    chosen_policy: str = 'min_collateral_with_min_benefit'
    chosen_target_fraction: float = 0.50  # of max feasible benefit PROXY; predefined, not chosen from labels
    coverage_targets: tuple = (0.50, 0.75)  # of max *predicted proxy* coverage, not labels
    benefit_cap_quantile: float = 0.999
    min_proxy_legitimate_fraction: float = 0.01
    # Conservative retrospective exposure controls: both cover ALL currencies.
    max_account_gross_usd: float | None = 100_000_000.0
    max_account_single_transfer_usd: float | None = 75_000_000.0
    max_portfolio_gross_usd: float | None = 250_000_000.0
    max_portfolio_proxy_collateral_usd: float | None = 15_000_000.0
    max_portfolio_touched_tx: int | None = 2500
    hub_outgoing_review: int = 10000
    exclude_extreme_hubs: bool = True
    mip_seconds: float = 5.0  # max per model; multiple models are tested
    mip_rel_gap: float = 0.015

SERVER_SHARED = Path('/home/ibm-user/shared')
SHARED = SERVER_SHARED if SERVER_SHARED.is_dir() else Path.cwd()
INPUT_PATH = SHARED / 'clean_transactions.csv'
RISK_PATH = SHARED / 'account_risk.csv'
PRIOR_PLAN_PATH = SHARED / 'freeze_plan.csv'
PRIOR_GUARDIAN_PATH = SHARED / 'guardian_candidate_outputs' / 'freeze_plan_candidate.csv'
OUTPUT_DIR = SHARED / 'guardian_v2_outputs'  # Never overwrites existing team's plan
CFG = Config()

print('Input transactions:', INPUT_PATH)
print('Input risk scores:', RISK_PATH)
print('New output directory:', OUTPUT_DIR)
print('SciPy mixed-integer solver:', HAVE_MILP)

# ===== ORIGINAL V2, CODE CELL 4 (read-time provenance patch only) =====
@dataclass
class Data:
    accounts: np.ndarray
    risks: np.ndarray
    sender: np.ndarray
    receiver: np.ndarray
    dollars: np.ndarray
    original_amount: np.ndarray
    currency: np.ndarray
    labelled: np.ndarray
    timestamp: np.ndarray
    from_bank: np.ndarray
    to_bank: np.ndarray
    tx_id: np.ndarray
    n_out: np.ndarray
    n_in: np.ndarray
    incident_gross: np.ndarray
    max_incident: np.ndarray
    tx_file_rows: int
    risk_file_rows: int
    fx_rates: dict
    read_provenance: dict | None = None


def incident_sum(f, t, w, n):
    '''Exact sum of weights touching each account; self-payments counted only once.'''
    ans = np.bincount(f, weights=w, minlength=n)
    other = f != t
    ans += np.bincount(t[other], weights=w[other], minlength=n)
    return ans


def load_data(transactions_path, risk_path, fx_rates=FX_TO_USD):
    transactions_path, risk_path = Path(transactions_path), Path(risk_path)
    if not transactions_path.is_file() or not risk_path.is_file():
        raise FileNotFoundError('Upload both clean_transactions.csv and account_risk.csv into /shared first.')
    required = {'tx_id', 'timestamp', 'from_acct', 'to_acct', 'amount_paid', 'currency', 'is_laundering'}
    optional = {'from_bank', 'to_bank', 'payment_format'}
    read_started = datetime.now(timezone.utc).isoformat()
    tx_sha_before = file_sha256(transactions_path)
    tx = pd.read_csv(transactions_path, usecols=lambda c: c in required | optional,
                     dtype={'from_acct': 'string', 'to_acct': 'string', 'currency': 'string',
                            'from_bank': 'string', 'to_bank': 'string'})
    if missing := sorted(required - set(tx.columns)):
        raise ValueError(f'Missing transaction columns: {missing}')
    for name in ('from_acct', 'to_acct', 'currency'):
        if tx[name].isna().any() or tx[name].astype('string').str.strip().eq('').any():
            raise ValueError(f'Blank/missing {name} values; do not auto-repair IDs or currency.')
    if tx['tx_id'].isna().any() or tx['tx_id'].duplicated().any():
        raise ValueError('Missing or duplicate tx_id: cannot safely evaluate distinct transfers.')
    raw_amt = pd.to_numeric(tx.amount_paid, errors='coerce').to_numpy(dtype=float)
    if not (np.isfinite(raw_amt).all() and (raw_amt >= 0).all()):
        raise ValueError('amount_paid has negative, missing or infinite values.')
    labels = pd.to_numeric(tx.is_laundering, errors='coerce').to_numpy(dtype=float)
    if not (np.isfinite(labels).all() and np.isin(labels, [0.,1.]).all()):
        raise ValueError('is_laundering must be 0 or 1 throughout.')
    timestamps = pd.to_datetime(tx.timestamp, errors='coerce', utc=True)
    if timestamps.isna().any():
        raise ValueError('Invalid timestamps. This notebook will not invent or forward-fill them.')
    found_currency = sorted(tx.currency.unique().tolist())
    missing_currency = sorted(set(found_currency) - set(fx_rates))
    if missing_currency:
        raise ValueError(f'No explicit FX assumption for: {missing_currency}; update FX_TO_USD or stop.')
    rates = tx.currency.map(fx_rates).to_numpy(dtype=float)
    if not (np.isfinite(rates).all() and (rates > 0).all()):
        raise ValueError('Invalid FX assumptions.')
    dollars = raw_amt * rates
    if not np.isfinite(dollars).all():
        raise ValueError('Converted values overflow; inspect large input amounts.')
    # Hash exactly the risk-file bytes that pandas parses. No pre/post timing ambiguity.
    risk_read_started = datetime.now(timezone.utc).isoformat()
    risk_snapshot = risk_path.read_bytes()
    risk_read_sha256 = hashlib.sha256(risk_snapshot).hexdigest()
    risk_snapshot_bytes = len(risk_snapshot)
    risk = pd.read_csv(io.BytesIO(risk_snapshot), usecols=['acct', 'risk_score'], dtype={'acct': 'string'})
    del risk_snapshot
    tx_sha_after = file_sha256(transactions_path)
    risk_sha_after = file_sha256(risk_path)
    if tx_sha_before != tx_sha_after or risk_read_sha256 != risk_sha_after:
        raise RuntimeError('Input files changed during Guardian V2 load: abort, coordinate before rerunning.')
    read_finished = datetime.now(timezone.utc).isoformat()
    read_provenance = {
        'transactions': {'path': str(transactions_path.resolve()), 'sha256_at_start_and_end': tx_sha_before,
                         'hash_before_equals_after': True, 'hash_method': 'SHA-256 before/after CSV parse'},
        'risk': {'path': str(risk_path.resolve()), 'sha256_exact_loaded_bytes': risk_read_sha256,
                 'byte_count': risk_snapshot_bytes, 'read_started_utc': risk_read_started,
                 'hash_method': 'SHA-256 of bytes parsed by pandas BytesIO',
                 'file_still_matches_snapshot': True},
        'overall_read_started_utc': read_started, 'overall_read_finished_utc': read_finished,
    }
    if risk.acct.isna().any() or risk.acct.duplicated().any():
        raise ValueError('Duplicate or null IDs in account_risk.csv')
    risk['risk_score'] = pd.to_numeric(risk.risk_score, errors='coerce')
    if risk.risk_score.isna().any() or not risk.risk_score.between(0,1).all():
        raise ValueError('Risk scores are missing or outside the interval [0, 1].')
    # Dense codes; only pseudonymous identifiers are used.
    from_account = tx.from_acct.to_numpy(dtype=str)
    to_account = tx.to_acct.to_numpy(dtype=str)
    accounts = np.unique(np.concatenate([from_account, to_account]))
    f = np.searchsorted(accounts, from_account).astype(np.int32)
    t = np.searchsorted(accounts, to_account).astype(np.int32)
    r = risk.set_index('acct').risk_score.reindex(accounts)
    coverage = r.notna().mean()
    if not r.notna().all():
        raise ValueError(f'Only {coverage:.3%} risk-score coverage; missing scores must not be zero-filled.')
    n = len(accounts)
    nout = np.bincount(f, minlength=n)
    nin = np.bincount(t, minlength=n)
    gross = incident_sum(f, t, dollars, n)
    max_incident = np.zeros(n, dtype=np.float64)
    np.maximum.at(max_incident, f, dollars)
    np.maximum.at(max_incident, t, dollars)
    send_bank = tx.from_bank.fillna('unknown').to_numpy(str) if 'from_bank' in tx else np.repeat('unknown', len(tx))
    recv_bank = tx.to_bank.fillna('unknown').to_numpy(str) if 'to_bank' in tx else np.repeat('unknown', len(tx))
    data = Data(accounts, r.to_numpy(float), f, t, dollars, raw_amt,
                tx.currency.to_numpy(str), labels.astype(np.int8), timestamps.to_numpy(),
                send_bank, recv_bank, tx.tx_id.to_numpy(), nout, nin, gross,
                max_incident, len(tx), len(risk), dict(fx_rates), read_provenance)
    del tx, risk
    print(f'ALL-CURRENCY data: {data.tx_file_rows:,} transfers; {n:,} accounts; risk coverage {coverage:.2%}; {len(found_currency)} currencies')
    print('Currencies:', ', '.join(found_currency))
    print('WARNING: Approximate FX assumptions and full-history risk scores; retrospective only.')
    return data

# ===== ORIGINAL V2, CODE CELL 6 (read-time provenance patch only) =====
@dataclass
class Problem:
    data: Data
    cfg: Config
    candidates: np.ndarray
    group_nodes: list
    group_benefit: np.ndarray
    group_cost: np.ndarray
    group_gross: np.ndarray
    group_count: np.ndarray
    risk_proxy: np.ndarray
    benefit_cap: float
    base_A: object
    base_lower: np.ndarray
    base_upper: np.ndarray
    eligible_count: int


def build_problem(d: Data, cfg: Config):
    if cfg.k < 1 or cfg.max_candidates < 1 or cfg.shortlist_each < 1:
        raise ValueError('Invalid action or shortlist budget.')
    if not (0 < cfg.benefit_cap_quantile <= 1 and 0 <= cfg.min_proxy_legitimate_fraction <= 1):
        raise ValueError('Invalid amount cap/floor assumptions.')
    f, t, n = d.sender, d.receiver, len(d.accounts)
    q = np.clip(d.risks[f] * d.risks[t], 0., 1.)
    cap = float(np.quantile(d.dollars, cfg.benefit_cap_quantile))
    b = np.minimum(d.dollars, cap) * q
    c = d.dollars * np.maximum(1. - q, cfg.min_proxy_legitimate_fraction)
    gross = d.dollars
    allowed = (d.risks >= cfg.min_account_score) & ((d.n_in + d.n_out) > 0)
    if cfg.max_account_gross_usd is not None:
        allowed &= d.incident_gross <= cfg.max_account_gross_usd
    if cfg.max_account_single_transfer_usd is not None:
        allowed &= d.max_incident <= cfg.max_account_single_transfer_usd
    if cfg.exclude_extreme_hubs:
        allowed &= d.n_out < cfg.hub_outgoing_review
    eligible = np.flatnonzero(allowed)
    if not len(eligible):
        print('ABSTAIN: no accounts pass the configured all-currency exposure safeguards.')
    # Distinct shortlist criteria; never consult true labels.
    ranks = {}
    base_b = incident_sum(f, t, b, n)
    base_c = incident_sum(f, t, c, n)
    def contribute(values, limit, weight=1.0):
        scores = values.copy()
        scores[~allowed] = -np.inf
        take = min(limit, len(eligible))
        if take <= 0: return
        # Stable sort by score (ties by account code), no random sampling.
        ii = np.argsort(-scores, kind='stable')[:take]
        for rank, x in enumerate(ii):
            if np.isfinite(scores[x]):
                ranks[int(x)] = ranks.get(int(x), 0.0) + weight / (rank + 1)
    for lam in cfg.candidate_lambdas:
        contribute(base_b - lam*base_c, cfg.shortlist_each)
    contribute(base_b, cfg.shortlist_each, 1.4)
    contribute(base_b / np.maximum(base_c, 1.), cfg.shortlist_each, 1.0)
    risk_order = np.full(n, -np.inf)
    risk_order[allowed] = d.risks[allowed]
    contribute(risk_order, cfg.shortlist_each, .5)
    candidates = np.asarray(sorted(ranks, key=lambda x: (-ranks[x], x))[:cfg.max_candidates], dtype=np.int32)
    m = len(candidates)
    # Group transfers exactly by which shortlist endpoints they touch.
    mapping = np.full(n, m, dtype=np.int32)  # m is sentinel for not shortlisted
    mapping[candidates] = np.arange(m, dtype=np.int32)
    first, second = mapping[f], mapping[t]
    low, high = np.minimum(first, second), np.maximum(first, second)
    valid = low < m
    encoded = low[valid].astype(np.int64) * (m + 1) + high[valid]
    codes, inv = np.unique(encoded, return_inverse=True)
    group_nodes = []
    for code in codes:
        x, y = int(code // (m+1)), int(code % (m+1))
        group_nodes.append((x,) if y == m or x == y else (x, y))
    G = len(codes)
    gb = np.bincount(inv, weights=b[valid], minlength=G)
    gc = np.bincount(inv, weights=c[valid], minlength=G)
    gg = np.bincount(inv, weights=gross[valid], minlength=G)
    gn = np.bincount(inv, minlength=G).astype(float)
    # A compact sparse MILP constraint matrix. Each group y is exactly OR(x endpoints).
    rows, cols, vals, lo, hi = [], [], [], [], []
    def add(items, lower=-np.inf, upper=np.inf):
        idx = len(lo)
        for cidx, coeff in items:
            rows.append(idx); cols.append(cidx); vals.append(coeff)
        lo.append(lower); hi.append(upper)
    add([(i,1.) for i in range(m)], upper=min(cfg.k, m))
    for g, nodes in enumerate(group_nodes):
        y = m + g
        for x in nodes:
            add([(y,1.),(x,-1.)], lower=0.)
        add([(y,1.)] + [(x,-1.) for x in nodes], upper=0.)
    A = coo_matrix((np.asarray(vals,float),(rows,cols)), shape=(len(lo),m+G)).tocsr()
    print(f'Eligibility: {len(eligible):,} / {n:,} accounts; shortlist {m}; coverage groups {G:,}')
    print(f'Benefit clipped above ${cap:,.0f} USD per transfer; collateral/gross NEVER clipped.')
    return Problem(d, cfg, candidates, group_nodes, gb, gc, gg, gn, q, cap, A,
                   np.asarray(lo,float), np.asarray(hi,float), len(eligible))

# ===== ORIGINAL V2, CODE CELL 8 (read-time provenance patch only) =====
def coverage_by_group(p, selected):
    chosen = set(int(i) for i in selected)
    selected_local = {j for j, i in enumerate(p.candidates) if int(i) in chosen}
    return np.fromiter((any(j in selected_local for j in nodes) for nodes in p.group_nodes),
                       count=len(p.group_nodes), dtype=bool)


def plan_numbers(p, selected):
    cover = coverage_by_group(p, selected)
    return (float(p.group_benefit[cover].sum()), float(p.group_cost[cover].sum()),
            float(p.group_gross[cover].sum()), int(p.group_count[cover].sum()))


def meets_limits(p, numbers):
    _, c, g, tcount = numbers
    cfg = p.cfg
    tol = 1e-7
    return ((cfg.max_portfolio_proxy_collateral_usd is None or c <= cfg.max_portfolio_proxy_collateral_usd*(1+tol))
        and (cfg.max_portfolio_gross_usd is None or g <= cfg.max_portfolio_gross_usd*(1+tol))
        and (cfg.max_portfolio_touched_tx is None or tcount <= cfg.max_portfolio_touched_tx))


def greedy_fallback(p, mode, lam=0., target=0., message='heuristic approximation'):
    m = len(p.candidates)
    rows = [[] for _ in range(m)]
    for g,nodes in enumerate(p.group_nodes):
        for i in nodes: rows[i].append(g)
    rows = [np.asarray(z, dtype=np.int32) for z in rows]
    covered = np.zeros(len(p.group_nodes), bool)
    chosen_local = []
    current_b = current_c = current_g = 0.
    current_n = 0
    for _ in range(min(p.cfg.k,m)):
        options = []
        for i, idx in enumerate(rows):
            if i in chosen_local: continue
            new_idx = idx[~covered[idx]]
            db = float(p.group_benefit[new_idx].sum())
            dc = float(p.group_cost[new_idx].sum())
            dg = float(p.group_gross[new_idx].sum())
            dn = int(p.group_count[new_idx].sum())
            if not meets_limits(p, (current_b+db,current_c+dc,current_g+dg,current_n+dn)):
                continue
            if mode == 'max_benefit': score = db
            elif mode == 'penalised': score = db - lam * dc
            else: score = db / max(dc,1.)
            options.append((score, -i, i, db, dc, dg, dn))
        if not options: break
        score, _, i, db, dc, dg, dn = max(options)
        if score <= 0: break
        chosen_local.append(i)
        covered[rows[i]] = True
        current_b += db; current_c += dc; current_g += dg; current_n += dn
        if mode == 'min_cost_target' and current_b + 1e-5 >= target: break
    chosen = p.candidates[np.asarray(chosen_local,dtype=np.int32)]
    nums = plan_numbers(p, chosen)
    if mode == 'min_cost_target' and nums[0] < target - max(1.,target*1e-5):
        return None
    assert meets_limits(p, nums)
    return dict(mode=mode, method='', selected=chosen, n_selected=len(chosen),
                proxy_benefit=nums[0], proxy_collateral=nums[1], gross_touched=nums[2],
                tx_touched=nums[3], status='HEURISTIC / NO OPTIMALITY CLAIM: '+message,
                status_code=-1, mip_gap=None, runtime_seconds=0.,
                penalty_weight=lam, target_proxy=target)


def solve(p, mode, lam=0., target=0.):
    m, G = len(p.candidates), len(p.group_nodes)
    if m == 0:
        if mode == 'min_cost_target' and target > 0: return None
        return dict(mode=mode, method='', selected=np.array([],dtype=np.int32), n_selected=0,
                    proxy_benefit=0., proxy_collateral=0., gross_touched=0., tx_touched=0,
                    status='ABSTAIN: no candidate meets the configured safeguards',
                    status_code=-2, mip_gap=None, runtime_seconds=0.,
                    penalty_weight=lam, target_proxy=target)
    if not HAVE_MILP:
        return greedy_fallback(p, mode, lam, target, 'SciPy MILP unavailable')
    # Coefficients in millions of USD to reduce numerical range.
    scale = 1_000_000.
    cvec = np.zeros(m+G, dtype=float)
    if mode == 'max_benefit':
        cvec[m:] = -p.group_benefit/scale
    elif mode == 'penalised':
        cvec[m:] = (-p.group_benefit+lam*p.group_cost)/scale
    elif mode == 'min_cost_target':
        cvec[m:] = (p.group_cost + 0.00001*p.group_gross)/scale
        cvec[:m] = 1e-4  # stable preference for fewer actions in ties
    else:
        raise ValueError('Unknown solve mode: '+mode)
    rows = [p.base_A]
    lows = [p.base_lower]
    highs = [p.base_upper]
    def add_limit(weight, value, relation, row_scale=scale):
        row = csr_matrix((weight/row_scale, (np.zeros(G,dtype=int),np.arange(m,m+G))),shape=(1,m+G))
        rows.append(row)
        if relation == 'max': lows.append(np.array([-np.inf])); highs.append(np.array([value/row_scale]))
        else: lows.append(np.array([value/row_scale])); highs.append(np.array([np.inf]))
    if p.cfg.max_portfolio_proxy_collateral_usd is not None:
        add_limit(p.group_cost, p.cfg.max_portfolio_proxy_collateral_usd, 'max')
    if p.cfg.max_portfolio_gross_usd is not None:
        add_limit(p.group_gross, p.cfg.max_portfolio_gross_usd, 'max')
    if p.cfg.max_portfolio_touched_tx is not None:
        # Count constraints stay in transaction units, not million-dollar units.
        add_limit(p.group_count, p.cfg.max_portfolio_touched_tx, 'max', row_scale=1.)
    if mode == 'min_cost_target':
        add_limit(p.group_benefit, target, 'min')
    A = vstack(rows, format='csr')
    low, high = np.concatenate(lows), np.concatenate(highs)
    start = time.perf_counter()
    try:
        result = milp(cvec, integrality=np.ones(m+G,dtype=np.int8),
                      bounds=Bounds(np.zeros(m+G),np.ones(m+G)),
                      constraints=LinearConstraint(A,low,high),
                      options={'presolve':True,'time_limit':p.cfg.mip_seconds,
                               'mip_rel_gap':p.cfg.mip_rel_gap})
    except (ValueError, RuntimeError) as exc:
        return greedy_fallback(p,mode,lam,target,'MILP error: '+str(exc)[:140])
    duration = time.perf_counter() - start
    if result.x is None:
        return greedy_fallback(p,mode,lam,target,'MILP no incumbent: '+str(result.message)[:100])
    local = np.flatnonzero(result.x[:m] > .5)
    selected = p.candidates[local]
    measured = plan_numbers(p, selected)
    if len(selected)>p.cfg.k or not meets_limits(p,measured):
        raise AssertionError('The solver returned a policy-violating plan; stop and inspect.')
    if mode == 'min_cost_target' and measured[0] < target - max(1.,target*1e-5):
        raise AssertionError('Solver returned a solution below the requested proxy target.')
    # Verify solver y variables independently from selected endpoint coverage.
    y = coverage_by_group(p,selected)
    if not np.allclose(result.x[m:], y.astype(float), atol=1e-5):
        raise AssertionError('Coverage variables disagree with endpoint union.')
    return dict(mode=mode,method='',selected=selected,n_selected=len(selected),
                proxy_benefit=measured[0],proxy_collateral=measured[1],
                gross_touched=measured[2],tx_touched=measured[3],
                status=str(result.message),status_code=int(result.status),
                mip_gap=(float(result.mip_gap) if getattr(result,'mip_gap',None) is not None else None),
                runtime_seconds=duration,penalty_weight=lam,target_proxy=target)


def run_frontier(p):
    start = time.perf_counter()
    points = []
    best = solve(p, 'max_benefit')
    if best is None: raise ValueError('No feasible plan, including the empty plan: check constraints.')
    best['method'] = 'Maximum proxy benefit within safety budgets'
    points.append(best)
    for lam in p.cfg.frontier_lambdas:
        sol = solve(p, 'penalised', lam=float(lam))
        if sol is None: continue
        sol['method'] = f'Proxy penalty {lam:g} (constrained)'
        points.append(sol)
    if p.cfg.chosen_lambda not in p.cfg.frontier_lambdas:
        sol = solve(p, 'penalised', lam=p.cfg.chosen_lambda)
        if sol is not None:
            sol['method'] = f'Proxy penalty {p.cfg.chosen_lambda:g} (constrained)'
            points.append(sol)
    max_b = best['proxy_benefit']
    if max_b > 0:
        for fraction in sorted(set(p.cfg.coverage_targets + (p.cfg.chosen_target_fraction,))):
            if fraction <= 0 or fraction > 1: raise ValueError('Invalid coverage target.')
            sol = solve(p, 'min_cost_target', target=max_b*fraction)
            if sol is None:
                print(f'No feasible incumbent for {fraction:.0%} of best proxy benefit; skip (not proof of infeasibility).')
                continue
            sol['method'] = f'Minimum proxy collateral at {fraction:.0%} of feasible proxy maximum'
            points.append(sol)
    if p.cfg.chosen_policy == 'min_collateral_with_min_benefit':
        target = max_b * p.cfg.chosen_target_fraction
        matches = [z for z in points if z['mode']=='min_cost_target' and abs(z['target_proxy']-target)<max(1.,target*1e-8)]
        if matches:
            chosen = matches[0]
        else:
            chosen = dict(best)
            chosen['method'] = 'FALLBACK: max proxy benefit under safety budgets'
            chosen['status'] = 'Target-minimisation yielded no feasible incumbent; using safe max-proxy baseline. '+best['status']
    elif p.cfg.chosen_policy == 'penalty':
        chosen = next(z for z in points if z['mode']=='penalised' and z['penalty_weight']==p.cfg.chosen_lambda)
    else: raise ValueError('chosen_policy must be min_collateral_with_min_benefit or penalty')
    print(f'Optimisation scenarios: {len(points)}; elapsed {time.perf_counter()-start:.1f}s')
    return chosen, points

# ===== ORIGINAL V2, CODE CELL 10 (read-time provenance patch only) =====
def selected_mask(d, selected):
    active = np.zeros(len(d.accounts), dtype=bool)
    if len(selected): active[np.asarray(selected,dtype=np.int32)] = True
    return active[d.sender] | active[d.receiver]


def retrospective_evaluate(d, selected):
    touched = selected_mask(d,selected)
    is_dirty = d.labelled.astype(bool)
    b = float(d.dollars[touched & is_dirty].sum())
    c = float(d.dollars[touched & ~is_dirty].sum())
    all_dirty = float(d.dollars[is_dirty].sum())
    bad_accounts = np.zeros(len(d.accounts), dtype=bool)
    bad_accounts[d.sender[is_dirty]] = True
    bad_accounts[d.receiver[is_dirty]] = True
    dirty_indices = np.flatnonzero(is_dirty)
    if len(dirty_indices):
        largest_dirty_idx = int(dirty_indices[np.argmax(d.dollars[dirty_indices])])
        remaining_total = all_dirty - d.dollars[largest_dirty_idx]
        remaining_blocked = b - (d.dollars[largest_dirty_idx] if touched[largest_dirty_idx] else 0.)
        pct_no_largest = 100*remaining_blocked/remaining_total if remaining_total > 0 else np.nan
    else: pct_no_largest = np.nan
    return {
        'observed_laundering_touched_usd':b,
        'observed_legitimate_touched_usd':c,
        'observed_legitimate_per_1m_laundering_usd':(1_000_000*c/b if b>0 else np.nan),
        'observed_laundering_value_coverage_pct':(100*b/all_dirty if all_dirty>0 else np.nan),
        'observed_laundering_transfer_count':int(np.sum(touched & is_dirty)),
        'observed_legitimate_transfer_count':int(np.sum(touched & ~is_dirty)),
        'observed_laundering_transfer_recall_pct':(100*np.sum(touched & is_dirty)/np.sum(is_dirty) if np.sum(is_dirty) else np.nan),
        'observed_laundering_value_pct_excluding_biggest_single_transfer':pct_no_largest,
        'observed_label_negative_frozen_accounts':int(np.sum(~bad_accounts[selected])),
        'largest_observed_legitimate_touched_usd':float(np.max(d.dollars[touched & ~is_dirty],initial=0.)),
        'largest_labelled_dirty_touched_usd':float(np.max(d.dollars[touched & is_dirty],initial=0.)),
    }


def meets_account_guards(p, selected):
    d, cfg = p.data, p.cfg
    nodes = np.asarray(selected, dtype=np.int32)
    if np.any(d.risks[nodes] < cfg.min_account_score): return False
    if cfg.max_account_gross_usd is not None and np.any(d.incident_gross[nodes] > cfg.max_account_gross_usd): return False
    if cfg.max_account_single_transfer_usd is not None and np.any(d.max_incident[nodes] > cfg.max_account_single_transfer_usd): return False
    if cfg.exclude_extreme_hubs and np.any(d.n_out[nodes] >= cfg.hub_outgoing_review): return False
    return True


def compare_baselines(d, p, prior_plan_path=None, prior_guardian_path=None):
    k = p.cfg.k
    n = len(d.accounts)
    total_degree = d.n_in+d.n_out
    b = np.minimum(d.dollars, p.benefit_cap)*p.risk_proxy
    base_b = incident_sum(d.sender,d.receiver,b,n)
    out = []
    for name,nodes in (
        ('Top account risk — all currency',np.argsort(-d.risks,kind='stable')[:k]),
        ('Highest degree — all currency',np.argsort(-total_degree,kind='stable')[:k]),
        ('Highest gross proxy flow — all currency',np.argsort(-base_b,kind='stable')[:k]),
    ):
        out.append(dict(method=name, selected=np.asarray(nodes,dtype=np.int32),
                        n_selected=len(nodes), origin_rows=len(nodes), unmatched_ids=0,
                        status='Unconstrained historical baseline — NOT required to obey v2 safeguards'))
    mapper = pd.Series(np.arange(n,dtype=np.int32),index=d.accounts)
    for fname, name in ((prior_plan_path,'Existing Step 3 plan — all currency'),
                        (prior_guardian_path,'Previous Guardian candidate — all currency')):
        if fname is None or not Path(fname).is_file(): continue
        prior = pd.read_csv(fname,usecols=['acct'])
        acct = prior.acct.dropna().astype(str).drop_duplicates().head(k)
        located = mapper.reindex(acct).dropna().to_numpy(dtype=np.int32)
        out.append(dict(method=name,selected=located,n_selected=len(located),
                        origin_rows=len(acct),unmatched_ids=len(acct)-len(located),
                        status='Retrospective matched-ID comparison; prior plan not rerun under v2 constraints'))
    return out


def make_recommendation_rows(p,selected):
    d = p.data
    if not len(selected):
        return pd.DataFrame(columns=['rank','acct','risk_score','est_money_blocked',
            'estimated_proxy_collateral','full_gross_incident_usd','marginal_gross_touched_usd',
            'outgoing_tx','incoming_tx','reason_facts','requires_human_approval'])
    # Explanation rank sorted by *model's* individual suspicious proxy, not by labels.
    raw_benefit = np.minimum(d.dollars,p.benefit_cap)*p.risk_proxy
    c = d.dollars * np.maximum(1-p.risk_proxy,p.cfg.min_proxy_legitimate_fraction)
    scores = incident_sum(d.sender,d.receiver,raw_benefit,len(d.accounts))
    ordering = sorted(map(int,selected),key=lambda idx:(-scores[idx],d.accounts[idx]))
    seen = np.zeros(len(d.dollars),bool)
    rows=[]
    for rank,node in enumerate(ordering,1):
        inc = (d.sender==node)|(d.receiver==node)
        added = inc & ~seen
        seen |= inc
        neigh = np.unique(np.r_[d.receiver[d.sender==node],d.sender[d.receiver==node]])
        cross = int(np.count_nonzero(inc&(d.from_bank!=d.to_bank)))
        marginal_b=float(raw_benefit[added].sum()); marginal_c=float(c[added].sum())
        marginal_g=float(d.dollars[added].sum())
        summary = (f'Risk score {d.risks[node]:.3f} (account-level; not transaction probability); '
          f'{int(inc.sum()):,} historical transfers across all currencies; {len(neigh):,} counterparties; '
          f'{cross:,} cross-bank transfers; {marginal_b:,.0f} capped USD suspicion proxy, '
          f'{marginal_c:,.0f} USD uncalibrated collateral proxy, '
          f'{marginal_g:,.0f} USD full incremental historical exposure; HUMAN REVIEW ONLY.')
        rows.append(dict(rank=rank,acct=d.accounts[node],risk_score=float(d.risks[node]),
                         est_money_blocked=marginal_b,estimated_proxy_collateral=marginal_c,
                         full_gross_incident_usd=float(d.incident_gross[node]),
                         marginal_gross_touched_usd=marginal_g,
                         outgoing_tx=int(d.n_out[node]),incoming_tx=int(d.n_in[node]),
                         reason_facts=summary,requires_human_approval=True))
    return pd.DataFrame(rows)


def by_currency_audit(d, selected):
    touched = selected_mask(d,selected)
    subset = pd.DataFrame({'currency':d.currency[touched],
                           'is_laundering':d.labelled[touched],
                           'amount_original':d.original_amount[touched],
                           'amount_converted_usd':d.dollars[touched]})
    if subset.empty:
        return pd.DataFrame(columns=['currency','is_laundering','transactions','amount_original','amount_usd'])
    return subset.groupby(['currency','is_laundering'],as_index=False).agg(
        transactions=('amount_original','size'),
        amount_original=('amount_original','sum'),
        amount_usd=('amount_converted_usd','sum'))


def audit_marginals_only(d, plan):
    seen=np.zeros(len(d.dollars),bool)
    out=[]
    lookup={acct:i for i,acct in enumerate(d.accounts)}
    for row in plan.itertuples(index=False):
        idx=lookup[row.acct]
        touched=(d.sender==idx)|(d.receiver==idx)
        marginal=touched & ~seen
        seen |= touched
        is_dirty = d.labelled==1
        out.append({'rank':row.rank,'acct':row.acct,
                    'observed_labelled_laundering_marginal_usd':float(d.dollars[marginal&is_dirty].sum()),
                    'observed_labelled_legitimate_marginal_usd':float(d.dollars[marginal&~is_dirty].sum())})
    return pd.DataFrame(out,columns=['rank','acct','observed_labelled_laundering_marginal_usd',
                                     'observed_labelled_legitimate_marginal_usd'])

# ===== ORIGINAL V2, CODE CELL 12 (read-time provenance patch only) =====
def self_tests():
    # One transfer from account 0 to itself must add exactly once.
    np.testing.assert_allclose(incident_sum(np.array([0]),np.array([0]),np.array([17.]),2),[17,0])
    # Accounts 0 and 1 selected touch only two distinct transfers, not three.
    f=np.array([0,0,2]);t=np.array([1,0,1]);a=np.array([10.,20.,30.])
    d=Data(np.array(['A','B','C']),np.array([.9,.9,.9]),f,t,a,a,
           np.array(['US Dollar']*3),np.array([0,1,1],np.int8),np.array(['2026-01-01']*3),
           np.array(['B1']*3),np.array(['B2']*3),np.arange(3),
           np.bincount(f,minlength=3),np.bincount(t,minlength=3),
           incident_sum(f,t,a,3),np.array([20.,30.,30.]),3,3,dict(FX_TO_USD))
    np.testing.assert_allclose(selected_mask(d,np.array([0,1])),[True,True,True])
    assert d.dollars[selected_mask(d,np.array([0,1]))].sum()==60.
    cfg=replace(Config(),max_account_gross_usd=None,max_account_single_transfer_usd=None,
                max_portfolio_gross_usd=100.,max_portfolio_proxy_collateral_usd=None,
                max_portfolio_touched_tx=None,min_account_score=0.)
    p=build_problem(d,cfg)
    before = (p.candidates.copy(),p.group_benefit.copy(),p.group_cost.copy())
    d2=replace(d,labelled=np.array([1,0,0],np.int8))
    p2=build_problem(d2,cfg)
    np.testing.assert_array_equal(before[0],p2.candidates)
    np.testing.assert_allclose(before[1],p2.group_benefit)
    np.testing.assert_allclose(before[2],p2.group_cost)
    # All selected retrospective totals count each transaction once.
    aa=retrospective_evaluate(d,np.array([0,1]));assert aa['observed_laundering_touched_usd']==50.
    # Max gross cap can't be evaded by applying the clipping rule.
    assert not meets_limits(p,(0.,0.,110.,0))
    # Original array includes two currencies; no addition without conversion.
    assert FX_TO_USD['Brazil Real']>0 and FX_TO_USD['Brazil Real']!=1.
    print('Self-tests passed: endpoint union, self-transfer, label independence, gross guard, FX definitions.')

self_tests()

# ===== ORIGINAL V2, CODE CELL 14 (read-time provenance patch only) =====
def file_sha256(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as fp:
        for buf in iter(lambda:fp.read(chunk_size),b''):
            h.update(buf)
    return h.hexdigest()


def run_experiment(tx_path, risk_path, outdir, cfg, old_plan=None, old_guardian=None):
    start=time.perf_counter()
    d=load_data(tx_path,risk_path)
    p=build_problem(d,cfg)
    chosen, scenarios=run_frontier(p)
    baselines=compare_baselines(d,p,old_plan,old_guardian)
    out=Path(outdir);out.mkdir(parents=True,exist_ok=True)
    rows=[]
    for sol in scenarios+baselines:
        observed=retrospective_evaluate(d,sol['selected'])
        if 'proxy_benefit' in sol:
            predicted=(sol['proxy_benefit'],sol['proxy_collateral'],sol['gross_touched'],sol['tx_touched'])
        else:
            # Baselines may violate new safety caps; report, do not disguise.
            selected=sol['selected']
            full=selected_mask(d,selected)
            prob=p.risk_proxy
            pred_b=np.minimum(d.dollars,p.benefit_cap)*prob
            pred_c=d.dollars*np.maximum(1-prob,cfg.min_proxy_legitimate_fraction)
            predicted=(float(pred_b[full].sum()),float(pred_c[full].sum()),
                       float(d.dollars[full].sum()),int(full.sum()))
        rows.append(dict(method=sol['method'],n_selected=len(sol['selected']),
                         selected_account_ids=';'.join(d.accounts[sol['selected']].tolist()),
                         proxy_benefit_usd=predicted[0],proxy_collateral_usd=predicted[1],
                         full_gross_touched_usd=predicted[2],tx_touched=predicted[3],
                         within_v2_limits=(meets_limits(p,predicted) and meets_account_guards(p,sol['selected']) and len(sol['selected'])<=cfg.k),
                         solver_status=sol.get('status',''),mip_gap=sol.get('mip_gap'),
                         runtime_seconds=sol.get('runtime_seconds'),
                         missing_prior_accounts=sol.get('unmatched_ids',0),
                         **observed))
    comparison=pd.DataFrame(rows)
    candidates=make_recommendation_rows(p,chosen['selected'])
    currencies=by_currency_audit(d,chosen['selected'])
    marginal=audit_marginals_only(d,candidates)
    comparison.to_csv(out/'all_currency_comparison_EVAL_ONLY.csv',index=False)
    candidates.to_csv(out/'freeze_plan_candidate.csv',index=False)
    currencies.to_csv(out/'selected_currency_audit_EVAL_ONLY.csv',index=False)
    marginal.to_csv(out/'candidate_label_marginals_EVAL_ONLY.csv',index=False)
    plan_recomputed=selected_mask(d,chosen['selected'])
    assert np.isclose(d.dollars[plan_recomputed].sum(), chosen['gross_touched'], rtol=1e-8,atol=0.1)
    assert len(candidates)==chosen['n_selected']
    assert len(candidates['acct'].unique())==chosen['n_selected']
    assert np.isclose(currencies.amount_usd.sum(),chosen['gross_touched'],rtol=1e-8,atol=0.1)
    if d.read_provenance is None or file_sha256(tx_path) != d.read_provenance['transactions']['sha256_at_start_and_end'] or file_sha256(risk_path) != d.read_provenance['risk']['sha256_exact_loaded_bytes']:
        raise RuntimeError('Source inputs changed since load; cannot certify this run.')
    out_manifest={
        'mode':'RETROSPECTIVE, SYNTHETIC DEVELOPMENT AUDIT; NOT PROSPECTIVE OR CAUSAL',
        'selection_uses_ground_truth_labels':False,
        'scores_are_calibrated_transaction_probabilities':False,
        'fx_pairing_requires_upstream_verification':'Confirm amount_paid means Amount Paid AND currency is Payment Currency',
        'usd_fx_rates_approximate':d.fx_rates,
        'dataset_rows':d.tx_file_rows,'risk_rows':d.risk_file_rows,
        'all_currencies_included':sorted(set(d.currency.tolist())),
        'input_hash_sha256':{'transactions':d.read_provenance['transactions']['sha256_at_start_and_end'],
                             'risk':d.read_provenance['risk']['sha256_exact_loaded_bytes']},
        'input_read_provenance':d.read_provenance,
        'config':asdict(cfg),'shortlisted_candidates':len(p.candidates),
        'candidate_restrictions_eligible_accounts':p.eligible_count,
        'coverage_groups':len(p.group_nodes),'chosen_method':chosen['method'],
        'chosen_solver_status':chosen['status'],'chosen_mip_gap':chosen['mip_gap'],
        'chosen_selected_count':len(chosen['selected']),
        'chosen_within_safety_limits':meets_limits(p,plan_numbers(p,chosen['selected'])),
        'scipy_milp_available':HAVE_MILP,'run_seconds':time.perf_counter()-start,
        'output_directory':str(out.resolve()),
        'no_freezes_executed':True,'manual_authorisation_required':True,
        'limitations':[
            'account risk scores are full-snapshot and may leak future information',
            'synthetic input slice may have been selected using ground-truth labels',
            'this is account-freeze coverage, not a tested targeted-payment-hold workflow',
            'amounts are historical transaction sums, not recovered proceeds or customer balances',
            'USD conversion uses time-invariant approximate FX rates',
            'no performance claims on an untouched chronological test period',
            'solver optimality is only within the shortlisted candidate set and proxy model',
            'collateral proxy is not a calibrated estimate of actual benign transaction value',
        ],
    }
    (out/'run_manifest.json').write_text(json.dumps(out_manifest,indent=2,default=str),encoding='utf-8')
    return dict(data=d,problem=p,chosen=chosen,scenarios=scenarios,comparison=comparison,
                freeze_plan=candidates,currency_audit=currencies,manifest=out_manifest,outdir=out)

In [ ]:
# ===== GUARDED RELEASE VERIFICATION (read-only inputs, new versioned output directory) =====
import argparse
import shutil
import sys
from collections import Counter


def _equal_usd(a, b):
    """Independent accounting checks tolerate sub-cent float accumulation only."""
    return bool(np.isclose(float(a), float(b), atol=0.05, rtol=1e-10, equal_nan=True))


def _record(checks, name, status, detail=''):
    assert status in ('PASS', 'FAIL', 'UNVERIFIED')
    checks.append({'check': name, 'status': status, 'detail': str(detail)})
    print(f'{status:10s} {name}: {detail}')


def _file_record(path):
    p=Path(path)
    return dict(path=str(p.resolve()), bytes=p.stat().st_size,
                sha256=file_sha256(p))


def _independent_accounting(d, names):
    # Deliberately do not reuse V2's selected_mask() / retrospective_evaluate().
    acct_names = np.asarray(names, dtype=str)
    active = np.isin(d.accounts, acct_names)
    sf = active[d.sender]
    st = active[d.receiver]
    touched = sf | st
    dirty = d.labelled == 1
    amount = d.dollars
    return {
        'selected_ids':acct_names.tolist(),
        'n_touched_transfers':int(np.count_nonzero(touched)),
        'both_endpoints_selected_count':int(np.count_nonzero(sf & st)),
        'self_transfers_touched_count':int(np.count_nonzero(touched & (d.sender==d.receiver))),
        'dirty_value_usd':float(np.sum(amount[touched & dirty])),
        'legitimate_value_usd':float(np.sum(amount[touched & ~dirty])),
        'gross_value_usd':float(np.sum(amount[touched])),
        'dirty_transfer_count':int(np.count_nonzero(touched & dirty)),
        'legitimate_transfer_count':int(np.count_nonzero(touched & ~dirty)),
    }


def _compare_alex(d, guardian_risk_path, alex_reference, alex_expected_hash, checks, provenance):
    expected_actual = d.read_provenance['risk']['sha256_exact_loaded_bytes']
    _record(checks, 'Risk file SHA-256 captured from EXACT bytes parsed by Guardian', 'PASS',
            f"{d.read_provenance['risk']['path']} | {expected_actual}")
    if alex_expected_hash:
        expected = str(alex_expected_hash).strip().lower()
        if len(expected) != 64 or any(c not in '0123456789abcdef' for c in expected):
            _record(checks,'Alex verified SHA-256 reference','FAIL','Invalid 64-character hex digest supplied')
        else:
            _record(checks,'Guardian risk SHA-256 matches Alex verified hash',
                    'PASS' if expected==expected_actual else 'FAIL',
                    'External supplied SHA-256 '+expected)
        provenance['alex_expected_sha256'] = expected
    else:
        _record(checks,'Guardian risk SHA-256 matches Alex verified hash','UNVERIFIED',
                'Ask Alex to supply independent verified SHA-256 of detector output')

    if alex_reference:
        ref=Path(alex_reference)
        if not ref.is_file():
            _record(checks,'Independent Alex score export comparison','FAIL', f'Missing reference file: {ref}')
        elif ref.resolve() == Path(guardian_risk_path).resolve():
            _record(checks,'Independent Alex score export comparison','UNVERIFIED',
                    'Reference is the same file Guardian already read; not independent verification')
        else:
            try:
                ref_raw=ref.read_bytes()
                ref_sha=hashlib.sha256(ref_raw).hexdigest()
                ref_df=pd.read_csv(io.BytesIO(ref_raw),usecols=['acct','risk_score'],dtype={'acct':'string'})
                ref_df['risk_score']=pd.to_numeric(ref_df.risk_score,errors='coerce')
                if ref_df.acct.isna().any() or ref_df.acct.duplicated().any() or ref_df.risk_score.isna().any():
                    raise ValueError('Reference contains null/duplicate acct or invalid scores')
                source=pd.read_csv(guardian_risk_path,usecols=['acct','risk_score'],dtype={'acct':'string'})
                source['risk_score']=pd.to_numeric(source.risk_score,errors='coerce')
                a=source.set_index('acct')['risk_score'].sort_index()
                b=ref_df.set_index('acct')['risk_score'].sort_index()
                same_ids=a.index.equals(b.index)
                same_scores=(same_ids and np.allclose(a.to_numpy(float),b.to_numpy(float),rtol=0,atol=1e-12))
                _record(checks,'Independent Alex score export comparison',
                        'PASS' if same_ids and same_scores else 'FAIL',
                        f'exact IDs={same_ids}; scores equal to 1e-12={same_scores}; reference SHA-256={ref_sha}')
                provenance['alex_reference']={'path':str(ref.resolve()),'sha256':ref_sha,
                                               'score_records':len(ref_df)}
            except Exception as e:
                _record(checks,'Independent Alex score export comparison','FAIL',type(e).__name__+': '+str(e))
    else:
        _record(checks,'Independent Alex score export comparison','UNVERIFIED',
                'No separately exported Alex score file supplied')


def audit_release(result, risk_path, transaction_path, output_dir,
                  alex_reference=None, alex_expected_hash=None, source_code_path=None,
                  original_v2_path=None, release_notebook_path=None):
    d=result['data']; p=result['problem']; chosen=result['chosen']
    plan=result['freeze_plan']; comp=result['comparison']; manifest=result['manifest']
    checks=[]
    provenance={
        'audit_type':'Golden Hour Guardian v2 independent final release checks',
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'source_input_at_load':d.read_provenance,
        'guardian_manifest_source_hashes':manifest.get('input_hash_sha256'),
        'input_snapshot_risk_sha256':d.read_provenance['risk']['sha256_exact_loaded_bytes'],
        'policy_name':manifest['chosen_method'],
        'selection_config':manifest['config'],
        'selected_account_ids_in_export':plan['acct'].astype(str).tolist(),
        'conventions':{
            'amount':'Approximate USD-normalised transaction value (not balances or recovered funds)',
            'benefit':'min(USD value, quantile cap) × sender account risk × receiver account risk',
            'collateral':'full USD value × max(1 − sender risk × receiver risk, minimum 1% proxy floor)',
            'selected_action':'simulated whole-account freeze recommendation, human review required',
            'ground_truth':'is_laundering permitted in retrospective audit ONLY',
            'optimality':'Only shortlisted candidates and specified proxy objective, subject to actual solver status/gap',
        },
    }
    if source_code_path and Path(source_code_path).is_file():
        provenance['release_script']=_file_record(source_code_path)
    if original_v2_path and Path(original_v2_path).is_file():
        provenance['source_original_v2_notebook']=_file_record(original_v2_path)
    if release_notebook_path and Path(release_notebook_path).is_file():
        provenance['executed_release_notebook']=_file_record(release_notebook_path)

    print('\n=== GUARDIAN V2 FINAL VERIFICATION ===')
    _record(checks,'Current transaction file unchanged since read',
            'PASS' if file_sha256(transaction_path)==d.read_provenance['transactions']['sha256_at_start_and_end'] else 'FAIL',
            d.read_provenance['transactions']['sha256_at_start_and_end'])
    _record(checks,'Current account_risk.csv unchanged since exact-byte read',
            'PASS' if file_sha256(risk_path)==d.read_provenance['risk']['sha256_exact_loaded_bytes'] else 'FAIL',
            d.read_provenance['risk']['sha256_exact_loaded_bytes'])
    _record(checks,'Recorded hashes agree with run manifest',
            'PASS' if manifest.get('input_hash_sha256')=={
                'transactions':d.read_provenance['transactions']['sha256_at_start_and_end'],
                'risk':d.read_provenance['risk']['sha256_exact_loaded_bytes']} else 'FAIL')
    print('Risk read path:',d.read_provenance['risk']['path'])
    print('Risk loaded byte hash:',d.read_provenance['risk']['sha256_exact_loaded_bytes'])
    print('Risk loaded started:',d.read_provenance['risk']['read_started_utc'])

    _compare_alex(d,risk_path,alex_reference,alex_expected_hash,checks,provenance)

    # Exact exported IDs and scores vs the detector file V2 itself consumed.
    id_to_score=dict(zip(d.accounts.tolist(),d.risks.tolist()))
    known=plan['acct'].isin(id_to_score)
    equal_score=(known.all() and np.allclose(plan.risk_score.to_numpy(float),
                                            plan.acct.map(id_to_score).to_numpy(float),atol=1e-12,rtol=0))
    _record(checks,'Exported account IDs and risk scores match Guardian loaded detector scores',
            'PASS' if known.all() and equal_score else 'FAIL',
            f'IDs found={int(known.sum())}/{len(plan)}; exact written scores match={equal_score}')
    _record(checks,'Detector scores complete, finite, and in [0,1]',
            'PASS' if len(d.accounts)==len(d.risks) and np.isfinite(d.risks).all() and ((d.risks>=0)&(d.risks<=1)).all() else 'FAIL')

    q=(d.risks[d.sender]*d.risks[d.receiver]).clip(0,1)
    benefit=np.minimum(d.dollars,p.benefit_cap)*q
    cost=d.dollars*np.maximum(1-q,p.cfg.min_proxy_legitimate_fraction)
    all_cur=set(d.currency.tolist())
    _record(checks,'All transaction currencies included and FX mapping finite',
            'PASS' if all_cur.issubset(d.fx_rates) and np.isfinite(d.dollars).all() else 'FAIL',
            f'{len(all_cur)} currencies')
    _record(checks,'Risk proxy, capped benefit and full-USD cost definitions',
            'PASS' if np.allclose(q,p.risk_proxy,atol=1e-12,rtol=0) and
                      np.isfinite(benefit).all() and np.isfinite(cost).all() and
                      np.all(cost>=0) else 'FAIL',
            f'benefit cap USD {p.benefit_cap:,.2f}; minimum cost fraction {p.cfg.min_proxy_legitimate_fraction:.2%}')

    selected_idx=np.asarray(chosen['selected'],dtype=np.int32)
    selected_names=d.accounts[selected_idx].tolist()
    chosen_by_name=comp.loc[comp['method']==manifest['chosen_method']]
    chosen_row=chosen_by_name.iloc[0] if len(chosen_by_name)==1 else None
    original_independent=_independent_accounting(d,selected_names)
    union=selected_mask(d,selected_idx)
    true_selected_gross=float(d.dollars[union].sum())
    direct_b=float(benefit[union].sum());direct_c=float(cost[union].sum());direct_t=int(union.sum())
    true_b=original_independent['dirty_value_usd'];true_c=original_independent['legitimate_value_usd']
    consistency=bool(chosen_row is not None and
        _equal_usd(chosen['proxy_benefit'],direct_b) and
        _equal_usd(chosen['proxy_collateral'],direct_c) and
        _equal_usd(chosen['gross_touched'],true_selected_gross) and
        _equal_usd(chosen_row['observed_laundering_touched_usd'],true_b) and
        _equal_usd(chosen_row['observed_legitimate_touched_usd'],true_c) and
        _equal_usd(chosen_row['proxy_benefit_usd'],direct_b) and
        _equal_usd(chosen_row['proxy_collateral_usd'],direct_c) and
        int(chosen_row['tx_touched'])==direct_t)
    _record(checks,'Independent all-currency value and proxy reconciliation for selected portfolio',
            'PASS' if consistency else 'FAIL',
            f'dirty={true_b:,.2f}; legitimate={true_c:,.2f}; full gross={true_selected_gross:,.2f}; overlap={original_independent["both_endpoints_selected_count"]}; touched={direct_t}')
    rank_unique=plan.acct.is_unique and plan['rank'].tolist()==list(range(1,len(plan)+1))
    same_plan_set=set(plan.acct)==set(selected_names)
    limits=bool(meets_account_guards(p,selected_idx) and meets_limits(p,plan_numbers(p,selected_idx)) and len(selected_idx)<=p.cfg.k)
    _record(checks,'Selected ranks/IDs unique and match chosen optimiser output',
            'PASS' if rank_unique and same_plan_set else 'FAIL',
            f'{len(plan)} accounts, budget={p.cfg.k}')
    _record(checks,'Selected portfolio satisfies configured account and portfolio budgets',
            'PASS' if limits else 'FAIL',
            f'gross={true_selected_gross:,.2f}; proxy collateral={direct_c:,.2f}; touched={direct_t}')

    # Independently verify every retrospective method, not merely the selected one.
    failed_methods=[]
    for r in comp.itertuples(index=False):
        ids=str(r.selected_account_ids).split(';') if str(r.selected_account_ids) not in ('','nan') else []
        ids=[a for a in ids if a]
        acct=_independent_accounting(d,ids)
        expected=(r.observed_laundering_touched_usd,r.observed_legitimate_touched_usd,
                  r.full_gross_touched_usd,r.observed_laundering_transfer_count,
                  r.observed_legitimate_transfer_count,r.tx_touched)
        actual=(acct['dirty_value_usd'],acct['legitimate_value_usd'],acct['gross_value_usd'],
                acct['dirty_transfer_count'],acct['legitimate_transfer_count'],acct['n_touched_transfers'])
        if (not all(_equal_usd(a,b) for a,b in zip(actual[:3],expected[:3])) or
            not all(int(a)==int(b) for a,b in zip(actual[3:],expected[3:]))):
            failed_methods.append(str(r.method))
    _record(checks,'No retrospective double-counting: every scenario recomputed with direct transfer OR',
            'PASS' if not failed_methods else 'FAIL',
            f'{len(comp)} scenarios verified; mismatched: '+str(failed_methods[:5]))
    audited_marginal=pd.read_csv(Path(output_dir)/'candidate_label_marginals_EVAL_ONLY.csv')
    sums=audited_marginal[['observed_labelled_laundering_marginal_usd',
                           'observed_labelled_legitimate_marginal_usd']].sum()
    marginals_ok=_equal_usd(sums.iloc[0],true_b) and _equal_usd(sums.iloc[1],true_c)
    _record(checks,'Incremental rank marginals sum to union-labelled totals exactly once',
            'PASS' if marginals_ok else 'FAIL')

    # Full decision pipeline rerun with ONLY the ground truth changed in memory.
    labels_original=d.labelled.copy()
    label_variants={
        'all_hidden':np.zeros(len(labels_original),dtype=np.int8),
        'shuffled':np.random.default_rng(1106).permutation(labels_original),
    }
    variants=[]
    for name,other_labels in label_variants.items():
        d_alt=replace(d,labelled=other_labels)
        p_alt=build_problem(d_alt,p.cfg)
        alt_chosen,alt_scenarios=run_frontier(p_alt)
        alt_plan=make_recommendation_rows(p_alt,alt_chosen['selected'])
        same=(np.array_equal(p.candidates,p_alt.candidates) and
              np.array_equal(p.group_benefit,p_alt.group_benefit) and
              np.array_equal(p.group_cost,p_alt.group_cost) and
              np.array_equal(p.group_gross,p_alt.group_gross) and
              np.array_equal(p.risk_proxy,p_alt.risk_proxy) and
              set(d.accounts[alt_chosen['selected']])==set(d.accounts[chosen['selected']]) and
              np.array_equal(alt_plan['acct'].to_numpy(),plan['acct'].to_numpy()) and
              np.allclose(alt_plan['est_money_blocked'],plan['est_money_blocked'],atol=1e-9,rtol=0) and
              np.allclose(alt_plan['estimated_proxy_collateral'],plan['estimated_proxy_collateral'],atol=1e-9,rtol=0) and
              alt_plan['reason_facts'].tolist()==plan['reason_facts'].tolist())
        variants.append((name,same))
        _record(checks,f'GROUND-TRUTH INDEPENDENCE — {name}',
                'PASS' if same else 'FAIL',
                'Same shortlist, complete proxy arrays, selected IDs, order, benefit/cost and explanation facts')

    label_free=not any('observed' in c.lower() or 'laundering' in c.lower() or 'labelled' in c.lower() or 'is_laundering' in c.lower()
                       for c in plan.columns)
    _record(checks,'Explanation-facing candidate CSV contains no ground-truth evaluation columns',
            'PASS' if label_free else 'FAIL',str(plan.columns.tolist()))

    required_baselines={'Top account risk — all currency','Highest degree — all currency',
                        'Highest gross proxy flow — all currency'}
    present=required_baselines.issubset(comp['method'].tolist())
    _record(checks,'Same-transaction-universe baseline reporting (top risk/degree/flow)',
            'PASS' if present and not failed_methods else 'FAIL',
            'Baselines cover all input currencies; NOT subject to identical V2 safety constraints')
    _record(checks,'Fair constrained comparison of baselines', 'UNVERIFIED',
            'V2 baselines share dataset and action count, but are not optimised under the same constraints')
    _record(checks,'No claim of actual funds recovered or customer balances',
            'PASS' if manifest.get('no_freezes_executed') and 'RETROSPECTIVE' in manifest.get('mode','')
              and 'EVAL_ONLY' in 'all_currency_comparison_EVAL_ONLY.csv' else 'FAIL',
            'Labels mark historical transfers that WOULD have been affected in the static simulation')
    code=chosen.get('status_code')
    gap=chosen.get('mip_gap')
    _record(checks,'Optimality claim within shortlist and solver tolerance',
            'PASS' if code==0 and gap is not None and gap <= p.cfg.mip_rel_gap+1e-6 else 'UNVERIFIED',
            f'solver status code={code}, gap={gap}; objective={chosen["mode"]}')
    _record(checks,'Alex training provenance and absence of future-information leakage',
            'UNVERIFIED','Requires Alex time-cutoff and training/prediction methodology; risk file matching cannot prove it')
    _record(checks,'FX source-field mapping independently verified with original dataset',
            'UNVERIFIED','Confirm Abdurub mapped amount_paid to Payment Currency; rate table remains approximate')
    _record(checks,'Prospective real-time interception or legally approved holds',
            'UNVERIFIED','This is a retrospective prototype and recommends review; no funds actually held')

    provenance['independent_portfolio_reconciliation']=original_independent
    provenance['solver']={'status':chosen['status'],'status_code':code,'gap':gap,
                          'shortlisted_candidate_count':len(p.candidates),
                          'eligible_account_count':p.eligible_count,'method':chosen['mode'],
                          'target_proxy':chosen.get('target_proxy'),'penalty_weight':chosen.get('penalty_weight')}
    provenance['label_independence_variants']={key:passed for key,passed in variants}
    provenance['checks']=checks

    out=Path(output_dir)
    output_fnames=['freeze_plan_candidate.csv','all_currency_comparison_EVAL_ONLY.csv',
                   'selected_currency_audit_EVAL_ONLY.csv','candidate_label_marginals_EVAL_ONLY.csv',
                   'run_manifest.json']
    provenance['output_files_at_release']={name:_file_record(out/name) for name in output_fnames}
    (out/'guardian_v2_provenance.json').write_text(json.dumps(provenance,indent=2,default=str),encoding='utf-8')

    counts=Counter(c['status'] for c in checks)
    status='FAIL' if counts['FAIL'] else ('UNVERIFIED' if counts['UNVERIFIED'] else 'PASS')
    report=[
        '# Golden Hour — Guardian V2 final verification',
        '',f'**Overall: {status}** — {counts["PASS"]} PASS, {counts["FAIL"]} FAIL, {counts["UNVERIFIED"]} UNVERIFIED.',
        '', '**Scope:** All-currency retrospective evidence on the exact files hashed at load. No action executed.',
        '', '**Guardian risk file loaded:** `'+d.read_provenance['risk']['path']+'`',
        '', '**Exact loaded risk SHA-256:** `'+d.read_provenance['risk']['sha256_exact_loaded_bytes']+'`',
        '', '**Transaction SHA-256:** `'+d.read_provenance['transactions']['sha256_at_start_and_end']+'`',
        '', '**Selected policy:** '+manifest['chosen_method'],
        '', '## Verification checks','',
        '| Status | Check | Evidence / qualification |','|---|---|---|',
    ]
    for c in checks:
        report.append('| '+c['status']+' | '+c['check'].replace('|','/')+' | '+c['detail'].replace('|','/').replace('\n',' ')+' |')
    report+=['','## Selection objective and exposure definitions','',
        'For each transaction, q = risk(sender) × risk(receiver) is an UNCALIBRATED proxy. '
        'Benefit = min(USD value, 99.9th percentile USD cap) × q. '
        'Collateral = full USD value × max(1 − q, 0.01). Gross exposure is never capped.',
        '',
        'The default policy minimises proxy collateral + 0.00001 × gross exposure with a tiny '
        'action-count tie preference, subject to at least 50% of the best FEASIBLE shortlisted proxy-benefit '
        'and the configured action-count, gross-exposure, collateral-proxy and transaction-count budgets. '
        'It does NOT maximise known true laundering value.',
        '',
        'Historical labelled laundering and legitimate sums are **transaction values touched by proposed account freezes**, '
        'not real money frozen, criminal proceeds recovered, financial balances, or measured losses.',
        '',
        'The optimised set is restricted to the scored shortlist and the chosen constraints. '
        'A solver optimum does not imply global optimality or safe real-world deployment.',
        '',
        '## Artifacts','',
        'Freeze candidates: `freeze_plan_candidate.csv`',
        'Evaluation: `all_currency_comparison_EVAL_ONLY.csv`',
        'Input/output hashes and settings: `guardian_v2_provenance.json` and `run_manifest.json`',
        'Candidate historical label audit: `candidate_label_marginals_EVAL_ONLY.csv` (NEVER feed to decision stage)',
        '',
        '## Release rule','',
        'Do not publish as fully independently verified unless Alex provides an independently verifiable '
        'detector hash/export AND model provenance/future-leakage concerns are resolved. '
        'Review all FAIL/UNVERIFIED items before tagging the GitHub release.',
    ]
    (out/'guardian_v2_verification_report.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
    print(f'\nVERIFICATION SUMMARY: {counts["PASS"]} PASS / {counts["FAIL"]} FAIL / {counts["UNVERIFIED"]} UNVERIFIED')
    print('Full report:',out/'guardian_v2_verification_report.md')
    return provenance


def run_release(shared='/home/ibm-user/shared', alex_reference=None, alex_expected_hash=None,
                original_v2_notebook=None, script_source=None, release_notebook_path=None):
    """Never writes/renames/removes original shared artifacts. Always creates a NEW folder."""
    shared=Path(shared).expanduser().resolve()
    tx_path=shared/'clean_transactions.csv'
    risk_path=shared/'account_risk.csv'
    if not tx_path.is_file() or not risk_path.is_file():
        raise FileNotFoundError(f'Need clean_transactions.csv and account_risk.csv under {shared}')
    prefix=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    out=shared/f'guardian_v2_final_verification_{prefix}_{hashlib.sha256(str(time.time_ns()).encode()).hexdigest()[:6]}'
    out.mkdir(parents=False,exist_ok=False)
    print('FINAL READ-ONLY INPUT VERIFICATION — outputs in NEW directory only:',out)
    print('Old /shared/freeze_plan.csv, verified notebooks, Ahmed outputs will NOT be overwritten.')
    previous=shared/'guardian_candidate_outputs'/'freeze_plan_candidate.csv'
    prior=shared/'freeze_plan.csv'
    result=run_experiment(tx_path,risk_path,out,Config(),
                          old_plan=prior if prior.is_file() else None,
                          old_guardian=previous if previous.is_file() else None)
    record=audit_release(result,risk_path,tx_path,out,alex_reference=alex_reference,
                         alex_expected_hash=alex_expected_hash,source_code_path=script_source,
                         original_v2_path=original_v2_notebook,
                         release_notebook_path=release_notebook_path)
    return dict(outdir=out,report=record,run=result)



## Final run on shared inputs (new output folder)

Fill Alex's separately confirmed source hash/reference if available. Do not invent a hash or compare the risk file against itself as independent validation.


In [ ]:
# Leave None until Alex independently provides the reference.
ALEX_REFERENCE_PATH = None   # e.g. '/home/ibm-user/shared/alex_verified_output.csv'
ALEX_EXPECTED_SHA256 = None # e.g. independent 64-character SHA-256 hash supplied by Alex

original = SHARED / 'golden_hour_03_freeze_v2.ipynb'
release_notebook = SHARED / 'guardian_v2_final_verification.ipynb'
release = run_release(
    shared=SHARED,
    alex_reference=ALEX_REFERENCE_PATH,
    alex_expected_hash=ALEX_EXPECTED_SHA256,
    original_v2_notebook=original if original.is_file() else None,
    release_notebook_path=release_notebook if release_notebook.is_file() else None,
    script_source=SHARED / 'guardian_v2_final_release.py'
        if (SHARED / 'guardian_v2_final_release.py').is_file() else None,
)
print('\nFINAL ARTIFACT FOLDER:', release['outdir'])
print('Open guardian_v2_verification_report.md and guardian_v2_provenance.json')


In [ ]:

from pathlib import Path
import pandas as pd

folder = Path("/home/ibm-user/shared/guardian_v2_outputs")

candidate = pd.read_csv(folder / "freeze_plan_candidate.csv")
explained = pd.read_csv(folder / "freeze_plan_explained.csv")

missing = set(candidate["acct"]) - set(explained["acct"])
extra = set(explained["acct"]) - set(candidate["acct"])

print("V2 accounts:", len(candidate))
print("Explained accounts:", len(explained))
print("Missing:", sorted(missing))
print("Extra:", sorted(extra))

assert len(candidate) == 10
assert len(explained) == len(candidate)
assert candidate["acct"].is_unique
assert explained["acct"].is_unique
assert not missing and not extra

assert "explanation" in explained.columns
assert explained["explanation"].notna().all()
assert explained["explanation"].astype(str).str.strip().ne("").all()

print("\nPASS — GUARDIAN V2 AND GRANITE ACCOUNT INTEGRATION")
